In [ ]:
import pandas as pd
df = pd.read_csv("/content/english_french.csv")

In [ ]:
df

,English,French
0,Go.,Va !
1,Go.,Marche.
2,Go.,En route !
3,Go.,Bouge !
4,Hi.,Salut !
...,...,...
229798,Death is something that we're often discourage...,La mort est une chose qu'on nous décourage sou...
229799,Since there are usually multiple websites on a...,Puisqu'il y a de multiples sites web sur chaqu...
229800,If someone who doesn't know your background sa...,Si quelqu'un qui ne connaît pas vos antécédent...
229801,It may be impossible to get a completely error...,Il est peut-être impossible d'obtenir un Corpu...


In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 229803 entries, 0 to 229802
Data columns (total 2 columns):
 #   Column   Non-Null Count   Dtype 
---  ------   --------------   ----- 
 0   English  229803 non-null  object
 1   French   229803 non-null  object
dtypes: object(2)
memory usage: 3.5+ MB


In [ ]:
df.duplicated().sum()

np.int64(0)

In [ ]:
df = df.rename(columns={
    "English": "english",
    "French": "french"
})

df.head()

,english,french
0,Go.,Va !
1,Go.,Marche.
2,Go.,En route !
3,Go.,Bouge !
4,Hi.,Salut !


In [ ]:
df["english_word_count"] = df["english"].astype(str).str.split().str.len()
df["french_word_count"] = df["french"].astype(str).str.split().str.len()

df.head()

,english,french,english_word_count,french_word_count
0,Go.,Va !,1,2
1,Go.,Marche.,1,1
2,Go.,En route !,1,3
3,Go.,Bouge !,1,2
4,Hi.,Salut !,1,2


In [ ]:
df[["english_word_count", "french_word_count"]].describe()

,english_word_count,french_word_count
count,229803.000000,229803.000000
mean,6.078063,6.630980
std,2.485588,2.815959
min,1.000000,1.000000
25%,4.000000,5.000000
50%,6.000000,6.000000
75%,7.000000,8.000000
max,55.000000,68.000000


In [ ]:
df = df[
    (df["english_word_count"] >= 1) &
    (df["english_word_count"] <= 10) &
    (df["french_word_count"] >= 1) &
    (df["french_word_count"] <= 12)
]

df.shape

(215659, 4)

In [ ]:
df_test = df.sample(150000, random_state=42).reset_index(drop=True)

In [ ]:
df_test

,english,french,english_word_count,french_word_count
0,That's already been done.,Ça a déjà été fait.,4,5
1,Tom finally left.,Tom est enfin parti.,3,4
2,The submarine has surfaced.,Le sous-marin est remonté à la surface.,4,7
3,Tom is in custody.,Tom est en garde à vue.,4,6
4,Corporations are competing to fill the vacuum.,Les entreprises se concurrencent pour remplir ...,7,8
...,...,...,...,...
149995,A dog is sometimes a dangerous animal.,Un chien peut être parfois un animal dangereux.,7,8
149996,I know how this looks to you.,Je sais comment cela vous semble.,7,6
149997,Every country has its own history.,Chaque pays a sa propre histoire.,6,6
149998,What do these markings mean?,Que signifient ces signaux ?,5,5


In [ ]:
import re

def clean(s):
    s = str(s).lower().strip()
    s = re.sub(r"([.!?,¿])", r" \1 ", s)   # split punctuation into its own token
    s = re.sub(r"\s+", " ", s).strip()
    return s

df_test["english"] = df_test["english"].apply(clean)
df_test["french"]  = df_test["french"].apply(clean)

In [ ]:
df_test["french_input"] = "<start> " + df_test["french"].astype(str)
df_test["french_target"] = df_test["french"].astype(str) + " <end>"

In [ ]:
df_test

,english,french,english_word_count,french_word_count,french_input,french_target
0,that's already been done .,ça a déjà été fait .,4,5,<start> ça a déjà été fait .,ça a déjà été fait . <end>
1,tom finally left .,tom est enfin parti .,3,4,<start> tom est enfin parti .,tom est enfin parti . <end>
2,the submarine has surfaced .,le sous-marin est remonté à la surface .,4,7,<start> le sous-marin est remonté à la surface .,le sous-marin est remonté à la surface . <end>
3,tom is in custody .,tom est en garde à vue .,4,6,<start> tom est en garde à vue .,tom est en garde à vue . <end>
4,corporations are competing to fill the vacuum .,les entreprises se concurrencent pour remplir ...,7,8,<start> les entreprises se concurrencent pour ...,les entreprises se concurrencent pour remplir ...
...,...,...,...,...,...,...
149995,a dog is sometimes a dangerous animal .,un chien peut être parfois un animal dangereux .,7,8,<start> un chien peut être parfois un animal d...,un chien peut être parfois un animal dangereux...
149996,i know how this looks to you .,je sais comment cela vous semble .,7,6,<start> je sais comment cela vous semble .,je sais comment cela vous semble . <end>
149997,every country has its own history .,chaque pays a sa propre histoire .,6,6,<start> chaque pays a sa propre histoire .,chaque pays a sa propre histoire . <end>
149998,what do these markings mean ?,que signifient ces signaux ?,5,5,<start> que signifient ces signaux ?,que signifient ces signaux ? <end>


In [ ]:
english_texts = df_test["english"].astype(str).tolist()
french_input_texts = df_test["french_input"].astype(str).tolist()
french_target_texts = df_test["french_target"].astype(str).tolist()

In [ ]:
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

eng_tokenizer = Tokenizer(filters="", lower=True, oov_token="<OOV>")
fr_tokenizer = Tokenizer(filters="", lower=True, oov_token="<OOV>")

eng_tokenizer.fit_on_texts(english_texts)
fr_tokenizer.fit_on_texts(french_input_texts + french_target_texts)

In [ ]:
encoder_sequences = eng_tokenizer.texts_to_sequences(english_texts)

decoder_input_sequences = fr_tokenizer.texts_to_sequences(french_input_texts)
decoder_output_sequences = fr_tokenizer.texts_to_sequences(french_target_texts)

In [ ]:
MAX_ENG_LEN = max(len(seq) for seq in encoder_sequences)
MAX_FR_LEN = max(len(seq) for seq in decoder_input_sequences)

print("Max English length:", MAX_ENG_LEN)
print("Max French length:", MAX_FR_LEN)

Max English length: 15
Max French length: 17


In [ ]:
encoder_input = pad_sequences(
    encoder_sequences,
    maxlen=MAX_ENG_LEN,
    padding="post"
)

decoder_input = pad_sequences(
    decoder_input_sequences,
    maxlen=MAX_FR_LEN,
    padding="post"
)

decoder_output = pad_sequences(
    decoder_output_sequences,
    maxlen=MAX_FR_LEN,
    padding="post"
)

In [ ]:
ENG_VOCAB_SIZE = len(eng_tokenizer.word_index) + 1
FR_VOCAB_SIZE = len(fr_tokenizer.word_index) + 1

print("English vocab size:", ENG_VOCAB_SIZE)
print("French vocab size:", FR_VOCAB_SIZE)

English vocab size: 14216
French vocab size: 27130


In [ ]:
from sklearn.model_selection import train_test_split

enc_train, enc_val, dec_in_train, dec_in_val, dec_out_train, dec_out_val = train_test_split(
    encoder_input,
    decoder_input,
    decoder_output,
    test_size=0.2,
    random_state=42
)

print(enc_train.shape)
print(enc_val.shape)
print(dec_in_train.shape)
print(dec_in_val.shape)
print(dec_out_train.shape)
print(dec_out_val.shape)

(120000, 15)
(30000, 15)
(120000, 17)
(30000, 17)
(120000, 17)
(30000, 17)


In [ ]:
import tensorflow as tf

BATCH_SIZE = 64

train_dataset = tf.data.Dataset.from_tensor_slices((
    enc_train,
    dec_in_train,
    dec_out_train
))

train_dataset = train_dataset.shuffle(10000).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

val_dataset = tf.data.Dataset.from_tensor_slices((
    enc_val,
    dec_in_val,
    dec_out_val
))

val_dataset = val_dataset.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

In [ ]:
for enc_batch, dec_in_batch, dec_out_batch in train_dataset.take(1):
    print("Encoder batch:", enc_batch.shape)
    print("Decoder input batch:", dec_in_batch.shape)
    print("Decoder output batch:", dec_out_batch.shape)

Encoder batch: (64, 15)
Decoder input batch: (64, 17)
Decoder output batch: (64, 17)


In [ ]:
import tensorflow as tf

EMBEDDING_DIM = 256
HIDDEN_UNITS = 512

print("English vocab:", ENG_VOCAB_SIZE)
print("French vocab:", FR_VOCAB_SIZE)

English vocab: 14216
French vocab: 27130


In [ ]:
class Encoder(tf.keras.Model):
    def __init__(self, vocab_size, embedding_dim, hidden_units):
        super().__init__()

        self.hidden_units = hidden_units

        self.embedding = tf.keras.layers.Embedding(
            vocab_size,
            embedding_dim,
            mask_zero=True
        )

        self.gru = tf.keras.layers.GRU(
            hidden_units,
            return_sequences=True,
            return_state=True
        )

    def call(self, x):
        x = self.embedding(x)

        encoder_outputs, encoder_state = self.gru(x)

        return encoder_outputs, encoder_state

In [ ]:
class BahdanauAttention(tf.keras.layers.Layer):
    def __init__(self, hidden_units):
        super().__init__()

        self.W1 = tf.keras.layers.Dense(hidden_units)
        self.W2 = tf.keras.layers.Dense(hidden_units)
        self.V = tf.keras.layers.Dense(1)

    def call(self, decoder_hidden, encoder_outputs):
        # decoder_hidden: (batch, hidden_units)
        # encoder_outputs: (batch, input_len, hidden_units)

        decoder_hidden_with_time_axis = tf.expand_dims(decoder_hidden, 1)

        score = self.V(
            tf.nn.tanh(
                self.W1(encoder_outputs) +
                self.W2(decoder_hidden_with_time_axis)
            )
        )

        attention_weights = tf.nn.softmax(score, axis=1)

        context_vector = attention_weights * encoder_outputs
        context_vector = tf.reduce_sum(context_vector, axis=1)

        return context_vector, attention_weights

In [ ]:
class Decoder(tf.keras.Model):
    def __init__(self, vocab_size, embedding_dim, hidden_units):
        super().__init__()

        self.hidden_units = hidden_units

        self.embedding = tf.keras.layers.Embedding(
            vocab_size,
            embedding_dim,
            mask_zero=True
        )

        self.attention = BahdanauAttention(hidden_units)

        self.gru = tf.keras.layers.GRU(
            hidden_units,
            return_sequences=True,
            return_state=True
        )

        self.fc = tf.keras.layers.Dense(vocab_size)

    def call(self, x, decoder_hidden, encoder_outputs):
        # x: (batch, 1)

        context_vector, attention_weights = self.attention(
            decoder_hidden,
            encoder_outputs
        )

        x = self.embedding(x)

        context_vector = tf.expand_dims(context_vector, 1)

        x = tf.concat([context_vector, x], axis=-1)

        output, state = self.gru(x)

        output = tf.reshape(output, (-1, output.shape[2]))

        prediction = self.fc(output)

        return prediction, state, attention_weights

In [ ]:
encoder = Encoder(
    vocab_size=ENG_VOCAB_SIZE,
    embedding_dim=EMBEDDING_DIM,
    hidden_units=HIDDEN_UNITS
)

decoder = Decoder(
    vocab_size=FR_VOCAB_SIZE,
    embedding_dim=EMBEDDING_DIM,
    hidden_units=HIDDEN_UNITS
)

In [ ]:
optimizer = tf.keras.optimizers.Adam()

loss_object = tf.keras.losses.SparseCategoricalCrossentropy(
    from_logits=True,
    reduction="none"
)

def loss_function(real, pred):
    mask = tf.math.logical_not(tf.math.equal(real, 0))

    loss = loss_object(real, pred)

    mask = tf.cast(mask, dtype=loss.dtype)

    loss *= mask

    return tf.reduce_mean(loss)

In [ ]:
@tf.function
def train_step(enc_input, dec_input, dec_output):
    loss = 0
    T = dec_input.shape[1]

    with tf.GradientTape() as tape:
        encoder_outputs, decoder_hidden = encoder(enc_input)

        decoder_input_word = dec_input[:, 0:1]            # <start>

        for t in range(T):
            predictions, decoder_hidden, _ = decoder(
                decoder_input_word, decoder_hidden, encoder_outputs
            )
            loss += loss_function(dec_output[:, t], predictions)

            if t + 1 < T:
                decoder_input_word = dec_input[:, t + 1:t + 2]   # teacher forcing

        batch_loss = loss / T

    variables = encoder.trainable_variables + decoder.trainable_variables
    gradients = tape.gradient(loss, variables)
    optimizer.apply_gradients(zip(gradients, variables))
    return batch_loss


@tf.function
def val_step(enc_input, dec_input, dec_output):
    loss = 0
    T = dec_input.shape[1]

    encoder_outputs, decoder_hidden = encoder(enc_input)
    decoder_input_word = dec_input[:, 0:1]

    for t in range(T):
        predictions, decoder_hidden, _ = decoder(
            decoder_input_word, decoder_hidden, encoder_outputs
        )
        loss += loss_function(dec_output[:, t], predictions)
        if t + 1 < T:
            decoder_input_word = dec_input[:, t + 1:t + 2]

    return loss / T

In [ ]:
import time

EPOCHS = 6

for epoch in range(EPOCHS):
    start = time.time()

    total_train_loss = 0
    total_val_loss = 0

    for batch, (enc_inp, dec_inp, dec_out) in enumerate(train_dataset):
        batch_loss = train_step(enc_inp, dec_inp, dec_out)
        total_train_loss += batch_loss

    for batch, (enc_inp, dec_inp, dec_out) in enumerate(val_dataset):
        batch_loss = val_step(enc_inp, dec_inp, dec_out)
        total_val_loss += batch_loss

    avg_train_loss = total_train_loss / len(train_dataset)
    avg_val_loss = total_val_loss / len(val_dataset)

    print(
        f"Epoch {epoch + 1} | "
        f"Train Loss: {avg_train_loss:.4f} | "
        f"Val Loss: {avg_val_loss:.4f} | "
        f"Time: {time.time() - start:.2f}s"
    )

Epoch 1 | Train Loss: 1.9779 | Val Loss: 1.5587 | Time: 284.87s
Epoch 2 | Train Loss: 1.2436 | Val Loss: 1.0549 | Time: 269.71s
Epoch 3 | Train Loss: 0.8027 | Val Loss: 0.8424 | Time: 269.22s
Epoch 4 | Train Loss: 0.5781 | Val Loss: 0.7641 | Time: 269.08s
Epoch 5 | Train Loss: 0.4450 | Val Loss: 0.7410 | Time: 269.29s
Epoch 6 | Train Loss: 0.3582 | Val Loss: 0.7364 | Time: 269.07s


In [ ]:
def translate(sentence):
    sentence = clean(sentence)

    sequence = eng_tokenizer.texts_to_sequences([sentence])

    sequence = tf.keras.preprocessing.sequence.pad_sequences(
        sequence,
        maxlen=MAX_ENG_LEN,
        padding="post"
    )

    encoder_outputs, encoder_hidden = encoder(sequence)

    decoder_hidden = encoder_hidden

    start_token = fr_tokenizer.word_index["<start>"]
    end_token = fr_tokenizer.word_index["<end>"]

    decoder_input_word = tf.expand_dims([start_token], 0)

    result = []

    attention_plot = []

    for _ in range(MAX_FR_LEN):
        predictions, decoder_hidden, attention_weights = decoder(
            decoder_input_word,
            decoder_hidden,
            encoder_outputs
        )

        predicted_id = tf.argmax(predictions[0]).numpy()

        if predicted_id == end_token:
            break

        predicted_word = fr_tokenizer.index_word.get(predicted_id, "")

        result.append(predicted_word)

        attention_plot.append(tf.squeeze(attention_weights).numpy())

        decoder_input_word = tf.expand_dims([predicted_id], 0)

    return " ".join(result)

In [ ]:
translate("you want to go out")

'tu veux sortir .'

In [ ]:
import os, json, shutil

os.makedirs("saved_model", exist_ok=True)

encoder.save_weights("saved_model/encoder.weights.h5")
decoder.save_weights("saved_model/decoder.weights.h5")

with open("saved_model/eng_tokenizer.json", "w") as f:
    f.write(eng_tokenizer.to_json())
with open("saved_model/fr_tokenizer.json", "w") as f:
    f.write(fr_tokenizer.to_json())

config = {
    "ENG_VOCAB_SIZE": ENG_VOCAB_SIZE,
    "FR_VOCAB_SIZE": FR_VOCAB_SIZE,
    "EMBEDDING_DIM": EMBEDDING_DIM,
    "HIDDEN_UNITS": HIDDEN_UNITS,
    "MAX_ENG_LEN": MAX_ENG_LEN,
    "MAX_FR_LEN": MAX_FR_LEN,
}
with open("saved_model/config.json", "w") as f:
    json.dump(config, f)

shutil.make_archive("saved_model", "zip", "saved_model")

from google.colab import files
files.download("saved_model.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>